<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Orthographic raster rendering of plans and sections from a point cloud.

All renders share one core: project points into a 2-D image plane, keep the
point nearest the viewer per pixel (z-buffer), then optionally

  * fill small gaps (nearest-neighbour within a radius) so sparse LiDAR
    slices read as surfaces,
  * draw the "cut face" (points within a thin band at the cut plane) in a
    solid colour on top, which gives the heavy wall lines of an
    architectural plan/section,
  * write a world file / DPI so the PNG is at true scale in CAD and print.

Coordinate conventions
  * Plan:    image x = world +X (east), image y (down) = world -Y (north up).
  * Section: image x = along the section line A->B, image y (down) = -Z.
           The viewer stands on the side of the line given by `side`
           (left/right when walking A->B) and looks toward the other side;
           points between the line and `depth` metres beyond it are drawn.

In [7]:
#| echo: false
#| output: asis
show_doc(RenderResult)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/render.py#L32){target="_blank" style="float:right; font-size:smaller"}

### RenderResult

```python
def RenderResult(
    image:Image.Image, pixel_size:float, origin:tuple[float, float], frame:dict, stats:dict=<factory>
)->None:
```

*RenderResult(image: 'Image.Image', pixel_size: 'float', origin: 'tuple[float, float]', frame: 'dict', stats: 'dict' = <factory>)*

In [14]:
#| echo: false
#| output: asis
show_doc(render_plan)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/render.py#L209){target="_blank" style="float:right; font-size:smaller"}

### render_plan

```python
def render_plan(
    cloud:Cloud, *, cut_height:float=1.2, level:float | None=None, below:float | None=None, look:str='down',
    pixel_size:float=0.005, style:Style='hybrid', bounds:tuple[float, float, float, float] | None=None,
    margin:float=0.25, fill_radius:float=0.02, cut_band:float=0.03, cut_colour:tuple=(0, 0, 0),
    cut_thicken:float=0.0, background:tuple=(255, 255, 255), max_pixels:int=60000000
)->RenderResult:
```

*Render a floor plan.*

cut_height  height of the cut plane above `level` (m).  Standard is 1.0–1.2.
level       world Z of the floor; default = detected floor.
below       how far below the cut plane to keep points (default: down to
            0.05 m below floor).  Use to hide lower storeys.
look        "down" (floor plan) or "up" (reflected ceiling plan; then
            cut_height is measured up from level and points ABOVE the cut are drawn).
bounds      (xmin, ymin, xmax, ymax) crop in world coords; default = whole cloud.
cut_band    thickness (m) of the slab at the cut plane drawn in cut_colour.
            0 disables.
style       color | depth | density | hybrid

In [16]:
#| echo: false
#| output: asis
show_doc(render_section)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/render.py#L264){target="_blank" style="float:right; font-size:smaller"}

### render_section

```python
def render_section(
    cloud:Cloud, *, a:tuple[float, float], b:tuple[float, float], depth:float=1.0, side:str='left',
    zmin:float | None=None, zmax:float | None=None, pixel_size:float=0.005, style:Style='hybrid', margin:float=0.15,
    fill_radius:float=0.02, cut_band:float=0.03, cut_colour:tuple=(0, 0, 0), cut_thicken:float=0.0,
    background:tuple=(255, 255, 255), max_pixels:int=60000000
)->RenderResult:
```

*Render a vertical section / elevation.*

a, b     endpoints of the section line in world XY (m).
side     which side of the line the viewer stands on when walking a -> b:
         "left" or "right".  The view looks across the line and shows
         everything from the line to `depth` metres on the far side.
depth    how far beyond the cut plane to draw (m).  Small (0.2) = pure
         section slice; large = section + elevation of what is behind.
zmin/zmax vertical crop; default = cloud floor-0.1 .. ceiling+0.1.

The image is a true (un-mirrored) view from the viewer's position:
side="right" -> a is on the image's left, b on the right;
side="left"  -> b is on the image's left, a on the right.
`frame["image_left_is"]` in the result records which.

In [18]:
#| echo: false
#| output: asis
show_doc(render_elevation)

---

[source](https://github.com/owen-AR/ptsrv/blob/main/ptsrv/render.py#L331){target="_blank" style="float:right; font-size:smaller"}

### render_elevation

```python
def render_elevation(
    cloud:Cloud, *, a, b, side:str='left', depth:float=50.0, **kw
)->RenderResult:
```

*Elevation = section with a large depth and no cut band.*

# Simple Render Test

In [ ]:
xyz = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
], dtype=np.float32)

rgb = np.full((len(xyz), 3), 255, dtype=np.uint8)

info = CloudInfo(
    name='synthetic',
    n_points=len(xyz),
    xmin=0.0, ymin=0.0, zmin=0.0,
    xmax=1.0, ymax=1.0, zmax=0.0,
    floor_z=0.0,
    ceiling_z=2.4,
    rotation_deg=0.0,
    source='synthetic test',
)

cloud = Cloud(xyz, rgb, info)
result = render_plan(cloud, cut_height=1.2, pixel_size=0.01)

assert result.png_bytes()